## 1. Helper: Embedding Generator with Fallback

If `llama-server` is running with embedding enabled (`--embedding` flag or embedding model loaded), we fetch embeddings directly from the server. If the server is offline during development, a deterministic mock embedding is generated so all math/visualizations remain interactive.


In [ ]:
def generate_embeddings_safe(texts: list[str]) -> np.ndarray:
    """Fetch embeddings from llama-server or fallback to deterministic mock vectors."""
    try:
        health = check_server_health()
        if health.get("status") == "online":
            return get_embeddings(texts)
    except Exception:
        pass
    
    # Deterministic fallback for offline testing
    print("[Note] Using synthetic normalized embeddings (start llama-server with --embedding for live model vectors)")
    vectors = []
    for text in texts:
        # Create a repeatable pseudo-vector based on string hash
        hash_val = hash(text) % (2**32)
        rng = np.random.RandomState(hash_val)
        v = rng.randn(384).astype(np.float32)
        # Add category clustering bias based on keywords
        if any(w in text.lower() for w in ["cat", "dog", "pet", "animal"]):
            v[0:50] += 2.0
        elif any(w in text.lower() for w in ["python", "code", "programming", "algorithm", "software"]):
            v[50:100] += 2.0
        elif any(w in text.lower() for w in ["space", "galaxy", "star", "planet", "telescope"]):
            v[100:150] += 2.0
        vectors.append(v)
    return normalize_vectors(np.array(vectors))


## 2. Sample Dataset & Embedding Generation

Let's define a corpus of sentences spanning different semantic domains (Animals, Programming, Astronomy).


In [ ]:
corpus = [
    # Animals
    "The domestic cat is a small carnivorous mammal.",
    "Dogs are loyal pets that enjoy outdoor games and fetching balls.",
    "A golden retriever is a popular breed of domestic dog.",
    "Felines are known for their agility and predatory instincts.",
    
    # Programming / Computer Science
    "Python is an interpreted, high-level programming language.",
    "NumPy provides support for large multi-dimensional arrays and matrices.",
    "Writing clean modular code improves software maintainability.",
    "Algorithms and data structures form the backbone of computer science.",
    
    # Astronomy / Space
    "The James Webb Space Telescope captures deep infrared images of distant galaxies.",
    "Stars generate energy through nuclear fusion in their cores.",
    "Mars is the fourth planet from the Sun and is known as the Red Planet.",
    "Black holes have gravitational pulls so strong that not even light can escape."
]

categories = ["Animals"] * 4 + ["Tech/Code"] * 4 + ["Space"] * 4

embeddings = generate_embeddings_safe(corpus)
print(f"Generated embeddings matrix shape: {embeddings.shape}")
print(f"Dimension of each embedding vector: {embeddings.shape[1]}")
print(f"L2 Norm of first vector (should be ~1.0): {np.linalg.norm(embeddings[0]):.4f}")


## 3. Pairwise Cosine Similarity Matrix

With normalized vectors ($\|v\|_2 = 1$), the cosine similarity matrix is simply the matrix product $M = V V^T$.


In [ ]:
sim_matrix = pairwise_similarity_matrix(embeddings)

# Plotting the heatmap
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(sim_matrix, cmap="viridis", vmin=0, vmax=1)

ax.set_xticks(range(len(corpus)))
ax.set_yticks(range(len(corpus)))
short_labels = [f"[{categories[i][:3]}] {c[:25]}..." for i, c in enumerate(corpus)]
ax.set_xticklabels(short_labels, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(short_labels, fontsize=8)

plt.colorbar(im, label="Cosine Similarity")
plt.title("Pairwise Semantic Similarity Heatmap", fontsize=13, pad=15)
plt.tight_layout()
plt.show()


## 4. Semantic Search Engine (Query vs Corpus)

Given a natural language query, we:
1. Embed the query $\vec{q}$
2. Compute similarity $\vec{s} = V \cdot \hat{q}$
3. Retrieve top-$K$ highest ranking documents using `top_k_similar`


In [ ]:
queries = [
    "How to write efficient Python software?",
    "Tell me about feline hunting and behavior",
    "Discoveries of planets and galaxies in outer space"
]

for query in queries:
    query_vec = generate_embeddings_safe([query])[0]
    top_indices, top_scores = top_k_similar(query_vec, embeddings, k=3)
    
    print("=" * 70)
    print(f"🔍 QUERY: '{query}'")
    print("=" * 70)
    for rank, (idx, score) in enumerate(zip(top_indices, top_scores), 1):
        print(f" Rank {rank} [Score: {score:.4f} | {categories[idx]}]: {corpus[idx]}")
    print()


## 5. 2D Dimensionality Reduction & Cluster Visualization (PCA)

Using Principal Component Analysis (PCA) to project high-dimensional embeddings into 2D space for visual clustering.


In [ ]:
pca = PCA(n_components=2)
reduced_vectors = pca.fit_transform(embeddings)

fig, ax = plt.subplots(figsize=(10, 7))
colors = {"Animals": "#e74c3c", "Tech/Code": "#3498db", "Space": "#2ecc71"}

for cat in set(categories):
    mask = [c == cat for c in categories]
    ax.scatter(
        reduced_vectors[mask, 0],
        reduced_vectors[mask, 1],
        c=colors[cat],
        label=cat,
        s=120,
        alpha=0.85,
        edgecolors="k"
    )

for i, txt in enumerate(corpus):
    ax.annotate(
        f"{corpus[i][:30]}...",
        (reduced_vectors[i, 0] + 0.02, reduced_vectors[i, 1] + 0.02),
        fontsize=8,
        alpha=0.85
    )

ax.set_title("2D Projection of Vector Embeddings (PCA)", fontsize=13)
ax.set_xlabel(f"PC 1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
ax.set_ylabel(f"PC 2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend(title="Semantic Category")
plt.tight_layout()
plt.show()
